# Video Face Swap Demo (inswapper + InsightFace)

**Logic:** 1 video mẫu (người chuyển động) + 1 ảnh khuôn mặt của bạn → video mới giữ nguyên chuyển động/nền của video gốc, chỉ thay khuôn mặt.

**Công nghệ dùng:**
- `insightface` (buffalo_l) để detect + align khuôn mặt từng frame
- `inswapper_128.onnx` để swap khuôn mặt
- `GFPGAN` (tùy chọn) để làm nét/phục hồi mặt sau khi swap
- `ffmpeg` để tách/ghép audio và dựng lại video

**Trước khi chạy:** Runtime > Change runtime type > chọn GPU (T4).

⚠️ Lưu ý: công nghệ face-swap có thể bị dùng sai mục đích (deepfake giả mạo người khác mà không có sự đồng ý). Chỉ dùng với ảnh/video của chính bạn hoặc người đã đồng ý, và cân nhắc gắn watermark/disclosure khi xuất bản sản phẩm thật.

## 1. Cài đặt thư viện

In [ ]:
!pip install -q insightface onnxruntime-gpu opencv-python-headless gfpgan basicsr facexlib
!apt-get -qq install -y ffmpeg > /dev/null

## 2. Tải model (inswapper + buffalo_l + GFPGAN)

Model `inswapper_128.onnx` không được host chính thức trên GitHub release nữa do vấn đề chính sách, 
nên bạn cần tự tải và upload lên Google Drive của mình, hoặc dùng link mirror cộng đồng (huggingface). 
Cell dưới thử tải từ 1 mirror phổ biến trên Hugging Face — nếu lỗi, bạn tải thủ công rồi upload vào `/content/`.

In [ ]:
import os
os.makedirs('/content/models', exist_ok=True)

# Mirror cộng đồng trên Hugging Face (kiểm tra lại link còn sống trước khi chạy)
!wget -q -O /content/models/inswapper_128.onnx https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx

# GFPGAN weights cho face restoration (tùy chọn)
!wget -q -O /content/models/GFPGANv1.4.pth https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth

print('Đã tải xong (kiểm tra dung lượng file bên dưới):')
!ls -lh /content/models/

## 3. Upload ảnh khuôn mặt của bạn + video mẫu

In [ ]:
from google.colab import files

print('>> Upload 1 ảnh khuôn mặt của bạn (rõ mặt, chính diện càng tốt):')
face_upload = files.upload()
source_face_path = list(face_upload.keys())[0]

print('\n>> Upload video mẫu (người chuyển động):')
video_upload = files.upload()
source_video_path = list(video_upload.keys())[0]

print(f'Ảnh mặt: {source_face_path}')
print(f'Video mẫu: {source_video_path}')

## 4. Khởi tạo model face analysis + face swapper

In [ ]:
import cv2
import insightface
from insightface.app import FaceAnalysis

app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

swapper = insightface.model_zoo.get_model('/content/models/inswapper_128.onnx', download=False)

# Lấy khuôn mặt nguồn (ảnh của bạn)
source_img = cv2.imread(source_face_path)
source_faces = app.get(source_img)
assert len(source_faces) > 0, 'Không tìm thấy khuôn mặt trong ảnh nguồn, thử ảnh khác rõ mặt hơn.'
source_face = source_faces[0]
print('Đã detect khuôn mặt nguồn thành công.')

## 5. (Tùy chọn) Khởi tạo GFPGAN để làm nét mặt sau swap

**Fix lỗi tương thích:** bản `torchvision` mới trên Colab đã xóa module `torchvision.transforms.functional_tensor` 
mà `basicsr` (dependency của GFPGAN) vẫn còn import theo đường cũ, gây lỗi `ModuleNotFoundError`. 
Cell dưới patch trực tiếp file trên đĩa bằng `sed` (không import basicsr trong Python nên không bị crash giữa chừng).

⚠️ **Nếu đã từng chạy lỗi ở cell GFPGAN trước đó trong session này**, hãy **Runtime > Restart session** rồi chạy lại từ đầu (kể cả cell cài đặt thư viện), vì Python có thể đã cache import bị lỗi. Sau khi patch xong, cell import GFPGAN bên dưới sẽ chạy được ngay lần đầu tiên.

In [ ]:
%%bash
# Tìm file degradations.py mà KHÔNG import basicsr (tránh crash trước khi patch được)
DEG_FILE=$(find / -path '*basicsr/data/degradations.py' 2>/dev/null | head -n1)

echo "File tìm thấy: $DEG_FILE"

sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' "$DEG_FILE"

grep -n "rgb_to_grayscale" "$DEG_FILE"

In [ ]:
USE_GFPGAN = True  # đổi thành False nếu muốn bỏ qua bước phục hồi mặt (nhanh hơn)

restorer = None
if USE_GFPGAN:
    from gfpgan import GFPGANer
    restorer = GFPGANer(
        model_path='/content/models/GFPGANv1.4.pth',
        upscale=1,
        arch='clean',
        channel_multiplier=2,
        bg_upsampler=None
    )

## 6. Xử lý video: swap mặt từng frame

In [ ]:
import os
from tqdm import tqdm

cap = cv2.VideoCapture(source_video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

output_noaudio = '/content/output_noaudio.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_noaudio, fourcc, fps, (width, height))

print(f'Video: {width}x{height} @ {fps:.1f}fps, {total_frames} frames')

for _ in tqdm(range(total_frames)):
    ret, frame = cap.read()
    if not ret:
        break

    target_faces = app.get(frame)
    result_frame = frame

    if len(target_faces) > 0:
        # Nếu video có nhiều người, mặc định swap người đầu tiên detect được
        target_face = target_faces[0]
        result_frame = swapper.get(frame, target_face, source_face, paste_back=True)

        if restorer is not None:
            _, _, result_frame = restorer.enhance(
                result_frame, has_aligned=False, only_center_face=False, paste_back=True
            )

    out.write(result_frame)

cap.release()
out.release()
print('Đã swap xong toàn bộ frame (chưa ghép audio).')

## 7. Ghép lại audio gốc vào video đã swap

In [ ]:
final_output = '/content/output_final.mp4'

!ffmpeg -y -i /content/output_noaudio.mp4 -i "{source_video_path}" \
  -c:v libx264 -crf 18 -preset fast \
  -map 0:v:0 -map 1:a:0? -shortest \
  {final_output}

print('Video hoàn chỉnh:', final_output)

## 8. Xem kết quả

In [ ]:
from IPython.display import Video
Video(final_output, embed=True, width=480)

In [ ]:
# Tải file về máy
from google.colab import files
files.download(final_output)

## Ghi chú / Hướng cải thiện tiếp theo

- **Flicker giữa các frame**: face swap từng frame độc lập nên đôi khi có giật/nhòe nhẹ theo thời gian. Có thể cải thiện bằng cách thêm temporal smoothing (trung bình landmark giữa các frame liền kề) hoặc dùng model chuyên video như **SimSwap** với chế độ video.
- **Nhiều khuôn mặt trong video**: hiện code chỉ swap khuôn mặt đầu tiên detect được mỗi frame. Nếu video có nhiều người, cần thêm logic track theo vị trí/ID khuôn mặt để chọn đúng người muốn thay.
- **Tốc độ**: xử lý frame-by-frame trên Colab free (T4) sẽ khá chậm với video dài. Nên test với video ngắn (5–10s) trước.
- **inswapper_128 model**: link tải có thể thay đổi do các vấn đề về chính sách/gỡ bỏ, nếu link trong notebook chết bạn cần tự tìm mirror khác hoặc lưu file vào Google Drive cá nhân rồi copy vào `/content/models/`.
- **Chất lượng ảnh mặt nguồn**: ảnh càng rõ, chính diện, ánh sáng đều thì kết quả swap càng tự nhiên.